# แบบฝึกหัด บทที่ 2 — ระเบียบวิธีประเมินผลระบบผู้แนะนำ
## 12 ข้อ + โจทย์ท้าทาย · มีเซลล์ตรวจคำตอบอัตโนมัติ

---

### วิธีใช้สมุดนี้

1. รัน **เซลล์เตรียมข้อมูล** (ส่วนที่ 0) ก่อนเสมอ — มีข้อมูล ตัวช่วย และระบบตรวจคำตอบอยู่ในนั้น
2. แต่ละข้อมีเซลล์ที่เขียนว่า `# TODO` ให้เติมโค้ดตรงนั้น
3. รันเซลล์ `ตรวจคำตอบ` ที่อยู่ถัดไป จะได้ ✅ หรือ ❌ ทันที
4. รันเซลล์สุดท้ายเพื่อดูคะแนนรวม

> **ห้ามใช้ไลบรารีสำเร็จรูป** เช่น `sklearn.metrics.ndcg_score` ในข้อ 1–12
> จุดประสงค์คือให้เขียนสูตรเองเพื่อให้เข้าใจว่าตัวชี้วัดแต่ละตัว "มองเห็น" อะไร
> ใช้ได้เพียง `numpy`, `pandas` และ Python มาตรฐาน

### แผนผังแบบฝึกหัด

| ข้อ | เรื่อง | ตรงกับหัวข้อในตำรา | ระดับ |
|---|---|---|---|
| 1 | MAE และ RMSE | 2.1 | ⭐ |
| 2 | ออกแบบกรณีที่ MAE เท่ากันแต่ RMSE ต่างกัน | 2.1 | ⭐⭐ |
| 3 | Precision@K · Recall@K · F1@K | 2.2 | ⭐ |
| 4 | Reciprocal Rank และ MRR@K | 2.3.1 | ⭐⭐ |
| 5 | Average Precision และ MAP@K | 2.3.2 | ⭐⭐ |
| 6 | DCG · IDCG · nDCG@K | 2.3.3 | ⭐⭐⭐ |
| 7 | AUC ด้วยการนับคู่ | 2.4 | ⭐⭐ |
| 8 | Catalog Coverage และ Gini ของการเปิดเผย | 2.5.1–2.5.2 | ⭐⭐ |
| 9 | Novelty และ LongTailRatio | 2.5.3–2.5.4 | ⭐⭐ |
| 10 | Calibration Error และ Parity Gap | 2.5.5–2.5.6 | ⭐⭐⭐ |
| 11 | Temporal Split และการตรวจการรั่วไหล | 2.8 | ⭐⭐⭐ |
| 12 | Sampled Negatives และ Winner Flip | 2.9 | ⭐⭐⭐ |
| ★ | โจทย์ท้าทาย: สร้างแบบจำลองของตัวเอง | 2.6 + 2.10 | ⭐⭐⭐⭐ |

**เฉลยอยู่ในไฟล์แยก** `Lab2_Exercises_Solutions.ipynb` — ลองทำเองให้ครบก่อนเปิดดู

---
## ส่วนที่ 0 — เตรียมข้อมูลและระบบตรวจคำตอบ (รันก่อนเสมอ)

In [ ]:
# ============================================================================
#  เซลล์เตรียมข้อมูล — รันเซลล์นี้ก่อนทำแบบฝึกหัดทุกข้อ
#  (ไม่ต้องอัปโหลดไฟล์ใด ๆ ข้อมูลทั้งหมดสร้างขึ้นในเซลล์นี้)
# ============================================================================
import numpy as np
import pandas as pd
from collections import Counter

pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
SEED = 2569

# ---------------------------------------------------------------- ระบบตรวจคำตอบ
_SCORE = {}


def check(label, got, expected, tol=1e-4, hint=""):
    """เทียบคำตอบกับค่าอ้างอิง  รองรับทั้งตัวเลขเดี่ยว ลิสต์ และ dict"""
    def close(a, b):
        if isinstance(b, dict):
            return set(a) == set(b) and all(close(a[k], b[k]) for k in b)
        if isinstance(b, (list, tuple, np.ndarray)):
            a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
            return a.shape == b.shape and np.allclose(a, b, atol=tol)
        if isinstance(b, (set, frozenset)):
            return set(a) == set(b)
        if isinstance(b, bool):
            return bool(a) is b
        try:
            return abs(float(a) - float(b)) <= tol
        except (TypeError, ValueError):
            return a == b

    try:
        ok = close(got, expected)
    except Exception as e:                       # noqa: BLE001
        ok, got = False, f"<ผิดพลาด: {e}>"
    _SCORE[label] = bool(ok)
    mark = "✅" if ok else "❌"
    def fmt(v):
        if isinstance(v, float):
            return f"{v:.4f}"
        if isinstance(v, (list, tuple, np.ndarray)):
            return "[" + ", ".join(f"{float(x):.4f}" for x in np.asarray(v).ravel()) + "]"
        return str(v)
    print(f"{mark}  {label:<46} ได้ {fmt(got)}"
          + ("" if ok else f"   ควรได้ {fmt(expected)}"))
    if not ok and hint:
        print(f"     💡 {hint}")
    return ok


def show_score():
    """สรุปคะแนนรวมของทั้งสมุด"""
    if not _SCORE:
        print("ยังไม่ได้ตรวจข้อใดเลย")
        return
    n_ok = sum(_SCORE.values())
    n = len(_SCORE)
    print("=" * 66)
    print(f"คะแนนรวม : {n_ok} / {n}  ({n_ok/n*100:.0f}%)")
    print("=" * 66)
    bad = [k for k, v in _SCORE.items() if not v]
    if bad:
        print("ข้อที่ยังไม่ผ่าน :")
        for b in bad:
            print("   -", b)
    else:
        print("ผ่านครบทุกข้อ 🎉")


# ---------------------------------------------------- ข้อมูลชุดที่ 1: ลิสต์อันดับ
# Model A และ B หยิบของถูกจำนวนเท่ากัน (3 ชิ้น) แต่วางคนละตำแหน่ง
RELEVANT = {"i2", "i5", "i9"}
RANK_A = ["i2", "i7", "i5", "i3", "i9", "i1", "i4", "i6", "i8", "i10"]
RANK_B = ["i7", "i3", "i1", "i2", "i4", "i5", "i6", "i8", "i9", "i10"]

# ---------------------------------------------- ข้อมูลชุดที่ 2: ผู้ใช้สามคน
# ใช้ในข้อ 4 และ 5
# ผู้ใช้แต่ละคนมีรายการที่ชอบจริงสามชิ้นเหมือนกันทั้งสองแบบจำลอง (rel_1, rel_2, rel_3)
# Model A และ B ต่างกันเพียง "ตำแหน่ง" ที่วางรายการทั้งสามนั้น
def _mk(user, positions, n=10):
    lst = [None] * n
    for idx, p in enumerate(sorted(positions), start=1):
        lst[p - 1] = f"{user}_rel{idx}"
    fill = 0
    for i in range(n):
        if lst[i] is None:
            fill += 1
            lst[i] = f"{user}_oth{fill}"
    return lst


_POS = {"Emma": ((1, 3, 5), (4, 6, 9)),
        "Liam": ((2, 4, 7), (1, 5, 8)),
        "Noah": ((2, 5, 6), (4, 7, 10))}
THREE_USERS = {}
for _u, (_pa, _pb) in _POS.items():
    THREE_USERS[_u] = {"A": _mk(_u, _pa), "B": _mk(_u, _pb),
                       "relevant": {f"{_u}_rel{i}" for i in (1, 2, 3)}}

# -------------------------------------------- ข้อมูลชุดที่ 3: MovieMini-12
MOVIE_META = pd.DataFrame([
    ("i1",  "The Matrix",    "ไซไฟ",      "ค่าย ก", 400),
    ("i2",  "Inception",     "ไซไฟ",      "ค่าย ก", 150),
    ("i3",  "Interstellar",  "ไซไฟ",      "ค่าย ก",  25),
    ("i4",  "Titanic",       "โรแมนติก",  "ค่าย ข", 300),
    ("i5",  "La La Land",    "โรแมนติก",  "ค่าย ข", 100),
    ("i6",  "The Notebook",  "โรแมนติก",  "ค่าย ง",   8),
    ("i7",  "John Wick",     "แอ็กชัน",   "ค่าย ค", 250),
    ("i8",  "Mad Max",       "แอ็กชัน",   "ค่าย ง",  15),
    ("i9",  "Parasite",      "ระทึกขวัญ", "ค่าย ข",  60),
    ("i10", "Toy Story",     "แอนิเมชัน", "ค่าย ข", 200),
    ("i11", "Coco",          "แอนิเมชัน", "ค่าย ค",  40),
    ("i12", "Spirited Away", "แอนิเมชัน", "ค่าย ง",   2),
], columns=["item_id", "item_name", "genre", "studio", "viewers"])

IDS = list(MOVIE_META.item_id)
GENRE = dict(zip(MOVIE_META.item_id, MOVIE_META.genre))
STUDIO = dict(zip(MOVIE_META.item_id, MOVIE_META.studio))
VIEWERS = dict(zip(MOVIE_META.item_id, MOVIE_META.viewers))
PLATFORM_USERS = 500          # ฐานผู้ใช้ทั้งแพลตฟอร์ม ใช้คำนวณ p(i)

USERS8 = ["Emma", "James", "Olivia", "Michael", "Sophia", "Daniel", "Ava", "Ethan"]
K5 = 5

# Model X : ยอดนิยมชุดเดียวให้ทุกคน | Model Y : เฉพาะบุคคล
MODEL_X = {u: ["i1", "i4", "i7", "i10", "i2"] for u in USERS8}
MODEL_Y = {
    "Emma":    ["i1", "i2", "i3", "i9", "i12"],
    "James":   ["i1", "i2", "i3", "i10", "i5"],
    "Olivia":  ["i4", "i5", "i6", "i11", "i9"],
    "Michael": ["i3", "i7", "i8", "i2", "i1"],
    "Sophia":  ["i4", "i5", "i6", "i10", "i12"],
    "Daniel":  ["i2", "i3", "i9", "i7", "i8"],
    "Ava":     ["i2", "i4", "i6", "i11", "i10"],
    "Ethan":   ["i1", "i5", "i7", "i3", "i11"],
}
USER_GROUP = {"ประวัติยาว": ["Emma", "Olivia", "Sophia", "Daniel"],
              "ประวัติสั้น": ["James", "Michael", "Ava", "Ethan"]}

# เฉลยความเกี่ยวข้อง: เรื่องที่ผู้ใช้ให้คะแนน ≥ 4
_R = np.array([
    [5, 5, 4, 1, np.nan, 1, 4, np.nan, 4, np.nan, np.nan, 3],
    [4, 5, 5, np.nan, 2, np.nan, 3, 4, np.nan, 2, np.nan, np.nan],
    [1, 2, np.nan, 5, 5, 4, np.nan, np.nan, 3, np.nan, 4, 4],
    [np.nan, 3, 4, 2, np.nan, np.nan, 5, 5, 4, np.nan, np.nan, np.nan],
    [2, np.nan, 1, 5, 5, 5, np.nan, 1, np.nan, 4, 5, np.nan],
    [np.nan, 4, 5, 3, 3, np.nan, 2, np.nan, 5, 4, 4, 5],
    [3, 4, np.nan, 4, np.nan, 3, np.nan, np.nan, 5, np.nan, 3, 4],
    [4, np.nan, 3, np.nan, 4, np.nan, 5, 4, np.nan, 5, np.nan, 5]], dtype=float)
RATINGS = pd.DataFrame(_R, index=USERS8, columns=IDS)
RELEVANT8 = {u: {i for i in IDS if RATINGS.loc[u, i] >= 4} for u in USERS8}
PROFILE8 = RELEVANT8            # ใช้ประวัติ = เรื่องที่ชอบ สำหรับข้อ 10

# ------------------------------------------- ข้อมูลชุดที่ 4: RSLab-Eval (ข้อ 11)
def make_interactions(n_users=400, n_items=200, n_weeks=20, seed=SEED):
    """ปฏิสัมพันธ์สังเคราะห์ที่ความนิยมเลื่อนตามเวลา (popularity drift)"""
    rng = np.random.default_rng(seed)
    items = np.array([f"it{n:03d}" for n in range(n_items)])
    base = rng.pareto(1.6, n_items) + 1
    base = base / base.sum()
    drift = rng.normal(0.0, 0.9, n_items)
    rows = []
    for w in range(n_weeks):
        pw = base * np.exp(drift * (w / max(1, n_weeks - 1)))
        pw = pw / pw.sum()
        for _ in range(int(rng.integers(400, 600))):
            u = int(rng.integers(0, n_users))
            i = int(rng.choice(n_items, p=pw))
            rows.append({"user_id": f"u{u:03d}", "item_id": items[i],
                         "rating": float(np.clip(round(rng.normal(3.6, 1.0)), 1, 5)),
                         "timestamp": w * 7 * 86400 + float(rng.uniform(0, 7 * 86400)),
                         "week": w})
    return (pd.DataFrame(rows).sort_values("timestamp")
            .drop_duplicates(subset=["user_id", "item_id"], keep="first")
            .reset_index(drop=True))


INTERACTIONS = make_interactions()

print("เตรียมข้อมูลเสร็จแล้ว\n")
print(f"  RELEVANT     : {sorted(RELEVANT)}")
print(f"  RANK_A       : {RANK_A}")
print(f"  RANK_B       : {RANK_B}")
print(f"  THREE_USERS  : {list(THREE_USERS)} (แต่ละคนมีลิสต์ของ Model A และ B)")
print(f"  MOVIE_META   : {len(MOVIE_META)} เรื่อง | MODEL_X, MODEL_Y : ผู้ใช้ {len(USERS8)} คน K = {K5}")
print(f"  INTERACTIONS : {len(INTERACTIONS):,} แถว "
      f"({INTERACTIONS.user_id.nunique()} ผู้ใช้ × {INTERACTIONS.item_id.nunique()} รายการ, "
      f"{INTERACTIONS.week.nunique()} สัปดาห์)")
print("\nพร้อมทำแบบฝึกหัดแล้ว — เริ่มที่ข้อ 1")

---
## ข้อ 1 — MAE และ RMSE ⭐
**หัวข้อ 2.1**

เขียนฟังก์ชัน `my_mae(y_true, y_pred)` และ `my_rmse(y_true, y_pred)` ด้วยตัวเอง

$$\text{MAE}=\frac{1}{n}\sum_{i}|\hat r_i-r_i| \qquad
\text{RMSE}=\sqrt{\frac{1}{n}\sum_{i}(\hat r_i-r_i)^2}$$

ทดสอบกับข้อมูลของผู้ใช้ Emma

| ภาพยนตร์ | คะแนนจริง | Model A ทาย |
|---|---|---|
| The Matrix | 5 | 4.2 |
| Inception | 5 | 4.6 |
| Interstellar | 4 | 4.3 |
| Titanic | 1 | 2.4 |
| The Notebook | 1 | 1.5 |

จากนั้นตอบคำถาม: **อัตราส่วน RMSE/MAE บอกอะไรเรา**

In [ ]:
# ============================== ข้อ 1 ==============================
def my_mae(y_true, y_pred) -> float:
    """ค่าเฉลี่ยความคลาดเคลื่อนสัมบูรณ์"""
    # TODO: เขียนสูตร MAE ที่นี่
    raise NotImplementedError


def my_rmse(y_true, y_pred) -> float:
    """รากที่สองของค่าเฉลี่ยความคลาดเคลื่อนกำลังสอง"""
    # TODO: เขียนสูตร RMSE ที่นี่
    raise NotImplementedError


# --------------------------- ตรวจคำตอบ ---------------------------
y_true = [5, 5, 4, 1, 1]
y_pred = [4.2, 4.6, 4.3, 2.4, 1.5]

check("1.1 MAE ของ Model A", my_mae(y_true, y_pred), 0.6800,
      hint="ผลรวม |e| = 3.40 หารด้วย 5")
check("1.2 RMSE ของ Model A", my_rmse(y_true, y_pred), 0.7874,
      hint="ผลรวม e² = 3.1000 หารด้วย 5 แล้วถอดรากที่สอง")
check("1.3 RMSE ≥ MAE เสมอ",
      all(my_rmse(a, b) >= my_mae(a, b) - 1e-12
          for a, b in [(np.random.rand(8) * 5, np.random.rand(8) * 5) for _ in range(200)]),
      True, hint="ผลจากอสมการของเยนเซน — ถ้าไม่ผ่าน แปลว่าสูตรใดสูตรหนึ่งผิด")

print(f"\nอัตราส่วน RMSE/MAE = {my_rmse(y_true, y_pred)/my_mae(y_true, y_pred):.4f}")
print("ยิ่งอัตราส่วนนี้สูง ยิ่งแปลว่าความผิดพลาดกระจุกอยู่ที่ไม่กี่รายการ")

---
## ข้อ 2 — ออกแบบกรณีที่ MAE เท่ากันแต่ RMSE ต่างกัน ⭐⭐
**หัวข้อ 2.1**

กำหนดคะแนนจริงของสี่รายการเท่ากับ `[4, 4, 4, 4]` ทั้งหมด

จงหา **ค่าพยากรณ์สองชุด** ที่

* ชุดที่ 1 `pred_even` — ให้ `MAE = 1.0` โดยผิดพลาด **เท่ากันทุกรายการ**
* ชุดที่ 2 `pred_spiky` — ให้ `MAE = 1.0` เท่ากันเป๊ะ แต่ `RMSE` **สูงกว่าชุดแรกอย่างน้อย 1.9 เท่า**

จากนั้นเขียนคำตอบว่าเหตุใด MAE จึงมองไม่เห็นความต่างนี้ (ลงในตัวแปร `answer_2`)

In [ ]:
# ============================== ข้อ 2 ==============================
y_true_2 = [4, 4, 4, 4]

# TODO: เติมค่าพยากรณ์สองชุด
pred_even = []       # MAE = 1.0 โดยผิดเท่ากันทุกรายการ
pred_spiky = []      # MAE = 1.0 เท่ากัน แต่ RMSE สูงกว่าอย่างน้อย 1.9 เท่า

answer_2 = "TODO: อธิบายว่าเหตุใด MAE จึงมองไม่เห็นความต่างนี้"

# --------------------------- ตรวจคำตอบ ---------------------------
if len(pred_even) == 4 and len(pred_spiky) == 4:
    m_even, r_even = my_mae(y_true_2, pred_even), my_rmse(y_true_2, pred_even)
    m_spiky, r_spiky = my_mae(y_true_2, pred_spiky), my_rmse(y_true_2, pred_spiky)
    print(f"ชุดที่ 1  {pred_even}   MAE = {m_even:.4f}  RMSE = {r_even:.4f}")
    print(f"ชุดที่ 2  {pred_spiky}   MAE = {m_spiky:.4f}  RMSE = {r_spiky:.4f}")
    print(f"อัตราส่วน RMSE ของสองชุด = {r_spiky/r_even:.2f} เท่า\n")
    check("2.1 MAE ของชุดที่ 1 เท่ากับ 1.0", m_even, 1.0)
    check("2.2 MAE ของชุดที่ 2 เท่ากับ 1.0", m_spiky, 1.0)
    check("2.3 RMSE ชุดที่ 2 สูงกว่าชุดที่ 1 อย่างน้อย 1.9 เท่า", r_spiky / r_even >= 1.9, True,
          hint="ลองให้ชุดที่ 2 ทายถูกสามรายการ แล้วพลาดหนักรายการเดียว")
    check("2.4 เขียนคำอธิบายแล้ว", not answer_2.startswith("TODO"), True)
else:
    print("❌ ยังไม่ได้เติม pred_even และ pred_spiky (ต้องมีชุดละ 4 ค่า)")

---
## ข้อ 3 — Precision@K · Recall@K · F1@K ⭐
**หัวข้อ 2.2**

เขียนสามฟังก์ชันด้วยตัวเอง แล้วคำนวณกับ `RANK_A` และ `RANK_B` ที่ `K = 10`

$$P@K=\frac{|\mathcal{R}_u \cap \mathcal{L}^K_u|}{K} \qquad
R@K=\frac{|\mathcal{R}_u \cap \mathcal{L}^K_u|}{|\mathcal{R}_u|} \qquad
F1@K=\frac{2PR}{P+R}$$

**คำถามที่ต้องตอบให้ได้** ทั้งสองแบบจำลองวางรายการที่ถูกคนละตำแหน่ง (A: 1, 3, 5 / B: 4, 6, 9)
แต่ตัวชี้วัดสามตัวนี้ให้ค่าเท่ากันหรือไม่ **เพราะอะไร**

In [ ]:
# ============================== ข้อ 3 ==============================
def my_precision_at_k(ranked, relevant, k):
    # TODO
    raise NotImplementedError


def my_recall_at_k(ranked, relevant, k):
    # TODO
    raise NotImplementedError


def my_f1_at_k(ranked, relevant, k):
    # TODO: ใช้ค่าเฉลี่ยฮาร์มอนิก และระวังกรณี P + R = 0
    raise NotImplementedError


# --------------------------- ตรวจคำตอบ ---------------------------
K = 10
pa = my_precision_at_k(RANK_A, RELEVANT, K)
pb = my_precision_at_k(RANK_B, RELEVANT, K)
ra = my_recall_at_k(RANK_A, RELEVANT, K)
f1a = my_f1_at_k(RANK_A, RELEVANT, K)

check("3.1 Precision@10 ของ A", pa, 0.3000)
check("3.2 Precision@10 ของ B", pb, 0.3000)
check("3.3 Recall@10 ของ A", ra, 1.0000, hint="ตัวส่วนคือจำนวนรายการที่ผู้ใช้ชอบจริง = 3")
check("3.4 F1@10 ของ A", f1a, 0.4615)
check("3.5 Precision@3 ของ A", my_precision_at_k(RANK_A, RELEVANT, 3), 0.6667)
check("3.6 Precision@3 ของ B", my_precision_at_k(RANK_B, RELEVANT, 3), 0.0000)
check("3.7 F1@10 ของ A เท่ากับของ B", abs(f1a - my_f1_at_k(RANK_B, RELEVANT, K)) < 1e-9, True)

print("\nสังเกต: ที่ K = 10 ทั้งสองแบบจำลองได้ค่าเท่ากันทุกตัว")
print("        แต่ที่ K = 3 แยกออกทันที — เพราะ K เล็กลงทำให้ 'ตำแหน่ง' เริ่มมีผล")

---
## ข้อ 4 — Reciprocal Rank และ MRR@K ⭐⭐
**หัวข้อ 2.3.1**

$$RR@K=\frac{1}{\text{อันดับของรายการเกี่ยวข้องชิ้นแรก}} \qquad
MRR@K=\frac{1}{|\mathcal{U}|}\sum_{u} RR@K(u)$$

ถ้าไม่พบรายการเกี่ยวข้องเลยใน Top-K ให้ `RR = 0`

ใช้ข้อมูล `THREE_USERS` (ผู้ใช้ 3 คน แต่ละคนมีลิสต์ของ Model A และ Model B)

| ผู้ใช้ | Model A วางของถูกที่อันดับ | Model B วางของถูกที่อันดับ |
|---|---|---|
| Emma | 1, 3, 5 | 4, 6, 9 |
| Liam | 2, 4, 7 | 1, 5, 8 |
| Noah | 2, 5, 6 | 4, 7, 10 |

**คำถามที่ต้องตอบให้ได้** RR มีจุดอ่อนสำคัญอะไร (ลองคิดถึงรายการที่ถูกชิ้นที่สองเป็นต้นไป)

In [ ]:
# ============================== ข้อ 4 ==============================
def my_reciprocal_rank(ranked, relevant, k):
    """ส่วนกลับของอันดับรายการเกี่ยวข้องชิ้นแรกใน Top-K  (ไม่พบเลย = 0)"""
    # TODO
    raise NotImplementedError


def my_mrr_at_k(model_key, users_data, k):
    """ค่าเฉลี่ยของ RR@K ข้ามผู้ใช้ทุกคน

    model_key  : "A" หรือ "B"
    users_data : dict แบบ THREE_USERS
    """
    # TODO
    raise NotImplementedError


# --------------------------- ตรวจคำตอบ ---------------------------
K = 10
print(f"{'ผู้ใช้':<8}{'RR ของ A':>12}{'RR ของ B':>12}")
print("-" * 32)
for u, d in THREE_USERS.items():
    print(f"{u:<8}{my_reciprocal_rank(d['A'], d['relevant'], K):>12.4f}"
          f"{my_reciprocal_rank(d['B'], d['relevant'], K):>12.4f}")
print()

check("4.1 RR ของ Emma กับ Model A",
      my_reciprocal_rank(THREE_USERS["Emma"]["A"], THREE_USERS["Emma"]["relevant"], K), 1.0000)
check("4.2 RR ของ Emma กับ Model B",
      my_reciprocal_rank(THREE_USERS["Emma"]["B"], THREE_USERS["Emma"]["relevant"], K), 0.2500)
check("4.3 MRR@10 ของ Model A", my_mrr_at_k("A", THREE_USERS, K), 0.6667)
check("4.4 MRR@10 ของ Model B", my_mrr_at_k("B", THREE_USERS, K), 0.5000)
check("4.5 ไม่พบรายการเกี่ยวข้องเลย -> RR = 0",
      my_reciprocal_rank(["n1", "n2", "n3"], {"r1"}, 3), 0.0000)
check("4.6 พบนอก Top-K -> RR = 0 (ตัดที่ K ก่อน)",
      my_reciprocal_rank(["n1", "n2", "r3"], {"r3"}, 2), 0.0000,
      hint="ต้องตัดลิสต์ให้เหลือ K ตำแหน่งก่อนค้นหา")

---
## ข้อ 5 — Average Precision และ MAP@K ⭐⭐
**หัวข้อ 2.3.2**

$$AP@K=\frac{1}{\min(K,|\mathcal{R}_u|)}\sum_{p=1}^{K} \text{rel}(p)\cdot P@p$$

คือ เฉลี่ยค่า Precision ณ **ทุกตำแหน่งที่พบรายการเกี่ยวข้อง**

**จุดที่พลาดกันบ่อยที่สุด** ตัวส่วนคือ $\min(K, |\mathcal{R}_u|)$ ไม่ใช่ $K$ และไม่ใช่จำนวนที่หาเจอ
เหตุผล: ถ้าผู้ใช้ชอบ 3 รายการ ก็เก็บได้สูงสุด 3 ชิ้น การหารด้วย 10 จะทำให้คะแนนต่ำเกินจริงเสมอ

**สิ่งที่ต้องทำเพิ่ม** เขียนฟังก์ชัน `ap_trace` ที่คืน *ลิสต์ของค่า Precision ณ จุดที่พบของถูก*
เพื่อให้ตรวจการคำนวณด้วยมือได้

In [ ]:
# ============================== ข้อ 5 ==============================
def ap_trace(ranked, relevant, k):
    """คืนลิสต์ของค่า Precision ณ ทุกตำแหน่งที่พบรายการเกี่ยวข้อง (ใน Top-K)

    ตัวอย่าง: RANK_A พบของถูกที่อันดับ 1, 3, 5 -> [1/1, 2/3, 3/5]
    """
    # TODO
    raise NotImplementedError


def my_average_precision_at_k(ranked, relevant, k):
    # TODO: ใช้ ap_trace แล้วหารด้วย min(k, |relevant|)
    raise NotImplementedError


def my_map_at_k(model_key, users_data, k):
    # TODO
    raise NotImplementedError


# --------------------------- ตรวจคำตอบ ---------------------------
K = 10
print("ร่องรอยการคำนวณ AP ของ Model A :", [round(x, 4) for x in ap_trace(RANK_A, RELEVANT, K)])
print("ร่องรอยการคำนวณ AP ของ Model B :", [round(x, 4) for x in ap_trace(RANK_B, RELEVANT, K)])
print()

check("5.1 ap_trace ของ A", ap_trace(RANK_A, RELEVANT, K), [1.0, 2/3, 0.6])
check("5.2 ap_trace ของ B", ap_trace(RANK_B, RELEVANT, K), [0.25, 1/3, 1/3])
check("5.3 AP@10 ของ A", my_average_precision_at_k(RANK_A, RELEVANT, K), 0.7556)
check("5.4 AP@10 ของ B", my_average_precision_at_k(RANK_B, RELEVANT, K), 0.3056)
check("5.5 MAP@10 ของ Model A", my_map_at_k("A", THREE_USERS, K), 0.5661)
check("5.6 MAP@10 ของ Model B", my_map_at_k("B", THREE_USERS, K), 0.3919)
check("5.7 ตัวส่วนใช้ min(K, |R|) ไม่ใช่ K",
      my_average_precision_at_k(["r1"], {"r1"}, 10), 1.0000,
      hint="ผู้ใช้ชอบ 1 รายการ และระบบวางไว้อันดับ 1 -> AP ต้องเท่ากับ 1.0 พอดี")

_apa = my_average_precision_at_k(RANK_A, RELEVANT, K)
_apb = my_average_precision_at_k(RANK_B, RELEVANT, K)
print(f"\nAP@10 ของ A สูงกว่า B {_apa/_apb:.2f} เท่า ทั้งที่ Precision@10 เท่ากันเป๊ะ")
print("-> AP ใช้ข้อมูลจากรายการเกี่ยวข้องทุกชิ้น ต่างจาก RR ที่สนใจเพียงชิ้นแรก")

---
## ข้อ 6 — DCG · IDCG · nDCG@K ⭐⭐⭐
**หัวข้อ 2.3.3**

$$DCG@K=\sum_{p=1}^{K}\frac{g(\text{rel}_p)}{\log_2(p+1)} \qquad
nDCG@K=\frac{DCG@K}{IDCG@K}$$

เขียนสามฟังก์ชัน

1. `my_dcg(gains, k, gain_type)` — รองรับทั้ง `"linear"` ($g=\text{rel}$) และ `"exponential"` ($g=2^{\text{rel}}-1$)
2. `my_idcg(relevance, k, gain_type)` — DCG ของลิสต์อุดมคติ (เรียง rel จากมากไปน้อย)
3. `my_ndcg_at_k(ranked, relevance, k, gain_type)`

**ข้อควรระวังสองข้อ**

* IDCG ขึ้นกับ **ผู้ใช้** ไม่ใช่ **แบบจำลอง** — A กับ B จึงหารด้วยค่าเดียวกัน
* ต้องประกาศ `gain_type` ทุกครั้งที่รายงาน เพราะสองแบบให้ค่าต่างกันมากเมื่อ rel > 1

In [ ]:
# ============================== ข้อ 6 ==============================
def my_dcg(gains, k, gain_type="linear"):
    """DCG@K จากลิสต์ของ rel ที่เรียงตามตำแหน่งจริงแล้ว"""
    # TODO
    raise NotImplementedError


def my_idcg(relevance, k, gain_type="linear"):
    """DCG ของลิสต์อุดมคติ  relevance เป็น dict {item: rel}"""
    # TODO
    raise NotImplementedError


def my_ndcg_at_k(ranked, relevance, k, gain_type="linear"):
    """nDCG@K  รายการที่ไม่อยู่ใน relevance ถือว่า rel = 0  ถ้า IDCG = 0 ให้คืน 0.0"""
    # TODO
    raise NotImplementedError


# --------------------------- ตรวจคำตอบ ---------------------------
K = 10
rel_map = {i: 1.0 for i in RELEVANT}

check("6.1 DCG@10 ของ A (rel ทวิภาค)",
      my_dcg([1, 0, 1, 0, 1, 0, 0, 0, 0, 0], K), 1.8869,
      hint="1/log2(2) + 1/log2(4) + 1/log2(6)")
check("6.2 IDCG@10 (ของถูก 3 ชิ้นอยู่อันดับ 1-3)", my_idcg(rel_map, K), 2.1309)
check("6.3 nDCG@10 ของ A", my_ndcg_at_k(RANK_A, rel_map, K), 0.8855)
check("6.4 nDCG@10 ของ B", my_ndcg_at_k(RANK_B, rel_map, K), 0.5105)
check("6.5 IDCG เท่ากันทั้งสองแบบจำลอง", my_idcg(rel_map, K), my_idcg(rel_map, K))
check("6.6 จัดอันดับสมบูรณ์แบบ -> nDCG = 1.0",
      my_ndcg_at_k(["i2", "i5", "i9", "i1", "i3"], rel_map, K), 1.0000)
check("6.7 gain แบบ exponential กับ rel หลายระดับ",
      my_dcg([3, 2, 0, 1], 4, "exponential"), 9.3235,
      hint="(2³−1)/log2(2) + (2²−1)/log2(3) + 0 + (2¹−1)/log2(5)")
check("6.8 IDCG = 0 -> คืน 0.0", my_ndcg_at_k(RANK_A, {}, K), 0.0000)

r = my_ndcg_at_k(RANK_A, rel_map, K) / my_ndcg_at_k(RANK_B, rel_map, K)
print(f"\nnDCG@10 ของ A สูงกว่า B {r:.2f} เท่า")
print("เทียบกับ MRR ที่ต่างกัน 4.00 เท่า -> ขนาดของความต่างขึ้นกับตัวชี้วัดที่เลือกด้วย")

---
## ข้อ 7 — AUC ด้วยการนับคู่ ⭐⭐
**หัวข้อ 2.4**

$$AUC=\frac{\text{จำนวนคู่ (เกี่ยวข้อง, ไม่เกี่ยวข้อง) ที่เรียงถูก}}{|\mathcal{R}_u|\times|\mathcal{N}_u|}$$

**ตีความ** ความน่าจะเป็นที่รายการเกี่ยวข้องซึ่งสุ่มมาหนึ่งชิ้น จะถูกจัดอันดับ *สูงกว่า*
รายการไม่เกี่ยวข้องซึ่งสุ่มมาหนึ่งชิ้น — ไม่ใช่ "ความแม่นยำ"

ในลิสต์ของเรามีของถูก 3 ชิ้น และของไม่ถูก 7 ชิ้น จึงมีคู่ทั้งหมด $3\times 7=21$ คู่

เขียน `my_auc(ranked, relevant)` และ `count_correct_pairs(ranked, relevant)`
แล้วตอบคำถาม: **AUC ของ Model B ต่ำกว่า 0.5 แปลว่าอะไร**

In [ ]:
# ============================== ข้อ 7 ==============================
def count_correct_pairs(ranked, relevant):
    """นับคู่ (เกี่ยวข้อง, ไม่เกี่ยวข้อง) ที่รายการเกี่ยวข้องอยู่ 'เหนือกว่า'"""
    # TODO
    raise NotImplementedError


def my_auc(ranked, relevant):
    """AUC จากการนับคู่  ถ้าไม่มีของถูกหรือไม่มีของผิดเลย ให้คืน float('nan')"""
    # TODO
    raise NotImplementedError


# --------------------------- ตรวจคำตอบ ---------------------------
check("7.1 จำนวนคู่ที่เรียงถูกของ A", count_correct_pairs(RANK_A, RELEVANT), 18)
check("7.2 จำนวนคู่ที่เรียงถูกของ B", count_correct_pairs(RANK_B, RELEVANT), 8)
check("7.3 AUC ของ A", my_auc(RANK_A, RELEVANT), 0.8571)
check("7.4 AUC ของ B", my_auc(RANK_B, RELEVANT), 0.3810)
check("7.5 เรียงถูกทุกคู่ -> AUC = 1.0",
      my_auc(["i2", "i5", "i9", "i1", "i3", "i4"], RELEVANT), 1.0000)
check("7.6 เรียงกลับด้านทั้งหมด -> AUC = 0.0",
      my_auc(["i1", "i3", "i4", "i2", "i5", "i9"], RELEVANT), 0.0000)

print(f"\nAUC ของ A = {my_auc(RANK_A, RELEVANT):.4f} -> สุ่มมาอย่างละหนึ่ง"
      f" มีโอกาส {my_auc(RANK_A, RELEVANT)*100:.2f}% ที่จะเรียงถูก")
print(f"AUC ของ B = {my_auc(RANK_B, RELEVANT):.4f} ซึ่ง 'ต่ำกว่าการเดาสุ่ม (0.5)'")
print("ข้อจำกัดที่ต้องจำ: AUC ให้น้ำหนักทุกคู่เท่ากัน ไม่ว่าอยู่ต้นหรือท้ายลิสต์")
print("จึงไม่เหมาะเป็น Primary Metric ของระบบที่แสดงผลเพียง 10 อันดับแรก")

---
## ข้อ 8 — Catalog Coverage และ Gini ของการเปิดเผย ⭐⭐
**หัวข้อ 2.5.1–2.5.2**

ใช้ `MODEL_X` (ยอดนิยมชุดเดียวให้ทุกคน) และ `MODEL_Y` (เฉพาะบุคคล) บนคลัง 12 เรื่อง
ผู้ใช้ 8 คน K = 5 → ช่องแสดงผลทั้งระบบ $8\times 5 = 40$ ช่อง

$$\text{Coverage@K}=\frac{|\bigcup_u \mathcal{L}^K_u|}{|\mathcal{I}|} \qquad
G=\frac{\sum_i (2i-n-1)\,x_i}{n\sum_i x_i}$$

**กฎบังคับสองข้อของ Gini ที่พลาดกันบ่อยที่สุด**

1. ต้อง **เรียง $x$ จากน้อยไปมากก่อน** เสมอ — ลืมเรียงแล้วสูตรคืนค่าที่ไม่มีความหมาย
2. รายการที่ **ไม่เคยถูกแนะนำเลยต้องอยู่ในเวกเตอร์ในฐานะ 0** — ถ้าตัดทิ้ง
   ค่าจะต่ำกว่าความจริง เพราะกลุ่มที่ถูกกีดกันที่สุดหายไปจากการคำนวณพอดี

**ตรวจทานง่าย ๆ** $\sum_i (2i-n-1)=0$ เสมอ และ $\sum_i x_i$ ต้องเท่ากับ 40 พอดี

In [ ]:
# ============================== ข้อ 8 ==============================
def exposure_vector(model_dict, all_items, users):
    """นับจำนวนครั้งที่แต่ละรายการถูกแนะนำ คืนลิสต์ยาวเท่า all_items (เรียงตาม all_items)

    รายการที่ไม่เคยถูกแนะนำเลยต้องมีค่า 0 อยู่ในลิสต์ด้วย
    """
    # TODO
    raise NotImplementedError


def my_coverage(model_dict, n_items, users):
    # TODO
    raise NotImplementedError


def my_gini(values):
    """สัมประสิทธิ์จีนี — อย่าลืมเรียงจากน้อยไปมากก่อน"""
    # TODO
    raise NotImplementedError


# --------------------------- ตรวจคำตอบ ---------------------------
ex_x = exposure_vector(MODEL_X, IDS, USERS8)
ex_y = exposure_vector(MODEL_Y, IDS, USERS8)
print("การเปิดเผยของ Model X :", ex_x, " รวม", sum(ex_x))
print("การเปิดเผยของ Model Y :", ex_y, " รวม", sum(ex_y))
print()

check("8.1 ผลรวมการเปิดเผยของ X = 40", sum(ex_x), 40)
check("8.2 ผลรวมการเปิดเผยของ Y = 40", sum(ex_y), 40)
check("8.3 ความยาวเวกเตอร์ = 12 (รวมรายการที่ได้ 0)", len(ex_x), 12,
      hint="รายการที่ไม่เคยถูกแนะนำต้องนับเป็น 0 ไม่ใช่ตัดทิ้ง")
check("8.4 Coverage@5 ของ Model X", my_coverage(MODEL_X, 12, USERS8), 0.4167)
check("8.5 Coverage@5 ของ Model Y", my_coverage(MODEL_Y, 12, USERS8), 1.0000)
check("8.6 Gini ของ Model X", my_gini(ex_x), 0.5833)
check("8.7 Gini ของ Model Y", my_gini(ex_y), 0.1500)
check("8.8 กระจายเท่ากันหมด -> Gini = 0", my_gini([5, 5, 5, 5]), 0.0000)
check("8.9 ต้องเรียงก่อนคำนวณ (สลับลำดับต้องได้ค่าเดิม)",
      my_gini([0, 8, 0, 3]), my_gini([8, 3, 0, 0]))

print(f"\nModel Y ครอบคลุมคลัง 100% ส่วน Model X เพียง 41.67% (มากกว่า 2.4 เท่า)")
print("และ Gini ของ Y ต่ำกว่ามาก -> กระจายโอกาสถูกค้นพบได้สม่ำเสมอกว่า")
print("แต่ Gini ต่ำไม่ได้แปลว่าแนะนำตรงใจกว่า ต้องอ่านคู่กับ Precision/nDCG เสมอ")

---
## ข้อ 9 — Novelty และ LongTailRatio ⭐⭐
**หัวข้อ 2.5.3–2.5.4**

$$\text{Novelty}@K=\frac{1}{K}\sum_{i\in\mathcal{L}^K_u}-\log_2 p(i),
\qquad p(i)=\frac{\text{ผู้เคยชม } i}{\text{ฐานผู้ใช้ทั้งแพลตฟอร์ม}}$$

ฐานผู้ใช้ทั้งแพลตฟอร์ม = `PLATFORM_USERS` = 500 คน · จำนวนผู้เคยชมอยู่ในคอลัมน์ `viewers`

**เส้นแบ่งหัว–หาง** ใช้เกณฑ์ *หนึ่งในสาม*: เรียงคลังตามความนิยมจากมากไปน้อย
แล้วไล่สะสมจนถึง 1/3 ของปฏิสัมพันธ์รวม รายการก่อนจุดนั้น = หัว ที่เหลือ = หาง

$$\text{LongTailRatio}@K=\frac{|\mathcal{L}^K_u \cap \text{Tail}|}{K}$$

**โจทย์เสริม** ข้อสุดท้ายให้ลอง "ปั่น" ค่า Novelty ของ Model X ให้สูงขึ้นอย่างน้อย 40%
โดย **เปลี่ยนรายการเพียงหนึ่งชิ้น** — เพื่อให้เห็นช่องโหว่ของตัวชี้วัดนี้ด้วยตัวเอง

In [ ]:
# ============================== ข้อ 9 ==============================
def my_novelty(reclist, viewers, n_platform_users):
    """ความใหม่เฉลี่ยเป็นบิต  ใช้ log ฐาน 2"""
    # TODO
    raise NotImplementedError


def my_head_tail_split(viewers, head_share=0.33):
    """คืน (head_items, tail_items) เป็น set ตามเกณฑ์ปฏิสัมพันธ์สะสมหนึ่งในสาม"""
    # TODO
    raise NotImplementedError


def my_long_tail_ratio(reclist, tail_items):
    # TODO
    raise NotImplementedError


# TODO: เปลี่ยนรายการเพียง 'หนึ่งชิ้น' จาก MODEL_X ให้ Novelty สูงขึ้นอย่างน้อย 40%
hacked_list = ["i1", "i4", "i7", "i10", "i2"]

# --------------------------- ตรวจคำตอบ ---------------------------
head, tail = my_head_tail_split(VIEWERS, 0.33)
print("กลุ่มหัว :", sorted(head), " | กลุ่มหาง :", len(tail), "เรื่อง")
nov_x = np.mean([my_novelty(MODEL_X[u], VIEWERS, PLATFORM_USERS) for u in USERS8])
nov_y = np.mean([my_novelty(MODEL_Y[u], VIEWERS, PLATFORM_USERS) for u in USERS8])
print(f"Novelty@5 เฉลี่ย : Model X = {nov_x:.4f} บิต | Model Y = {nov_y:.4f} บิต\n")

check("9.1 กลุ่มหัวมี 2 เรื่อง (i1, i4)", head, {"i1", "i4"},
      hint="ปฏิสัมพันธ์รวม 1,550 หนึ่งในสาม = 516.67")
check("9.2 Novelty@5 ของ Model X (ทุกคนได้ชุดเดียวกัน)", nov_x, 1.0236)
check("9.3 Novelty@5 ของ Michael กับ Model Y",
      my_novelty(MODEL_Y["Michael"], VIEWERS, PLATFORM_USERS), 2.4879)
check("9.4 Novelty@5 เฉลี่ยของ Model Y", nov_y, 2.8525)
check("9.5 LongTail@5 เฉลี่ยของ Model X",
      np.mean([my_long_tail_ratio(MODEL_X[u], tail) for u in USERS8]), 0.6000)
check("9.6 LongTail@5 เฉลี่ยของ Model Y",
      np.mean([my_long_tail_ratio(MODEL_Y[u], tail) for u in USERS8]), 0.8250)

nov_hack = my_novelty(hacked_list, VIEWERS, PLATFORM_USERS)
check("9.7 ปั่น Novelty ขึ้น ≥ 40% โดยเปลี่ยนรายการเดียว",
      (nov_hack / nov_x >= 1.40) and (len(set(hacked_list) - set(MODEL_X["Emma"])) == 1),
      True, hint="ลองแทนรายการที่ดังที่สุดด้วยเรื่องที่แทบไม่มีใครดู")
print(f"\nลิสต์ที่ปั่นแล้วได้ Novelty = {nov_hack:.4f} บิต ({(nov_hack/nov_x-1)*100:+.1f}%)")
print("บทเรียน: Novelty ถูกปั่นได้ด้วยรายการเดียว จึงห้ามใช้เป็น Primary Metric เดี่ยว ๆ")

---
## ข้อ 10 — Calibration Error และ Parity Gap ⭐⭐⭐
**หัวข้อ 2.5.5–2.5.6**

$$\text{CalibrationError}=\frac{1}{2}\sum_{c}\left|q(c)-p(c)\right|$$

* $p(c)$ = สัดส่วนแนว $c$ ใน **ประวัติ** ของผู้ใช้ (ใช้ `PROFILE8[u]` = เรื่องที่เขาให้คะแนน ≥ 4)
* $q(c)$ = สัดส่วนแนว $c$ ใน **รายการที่แนะนำ**

$$\text{ParityGap}=\max_g \overline{\text{metric}}_g-\min_g \overline{\text{metric}}_g$$

ใช้ `Precision@5` เป็นตัวชี้วัด และแบ่งกลุ่มด้วย `USER_GROUP` (ประวัติยาว / ประวัติสั้น)

> **กับดัก Leveling Down** ระบบที่แย่เท่ากันกับทุกคนจะได้ `ParityGap = 0` "สมบูรณ์แบบ"
> ขณะที่ระบบที่ดีกว่าใน *ทุกกลุ่ม* อาจได้ค่าสูงกว่า
> เกณฑ์ตายตัวแบบ "Gap ≤ 0.05" จะปฏิเสธระบบที่ดีกว่าทันที
> → **ต้องรายงานค่าเฉลี่ยรายกลุ่มควบคู่กับช่องว่างเสมอ**

In [ ]:
# ============================== ข้อ 10 ==============================
def genre_distribution(items, genre_of):
    """คืน dict {แนว: สัดส่วน} ของลิสต์รายการที่ให้มา (ถ้าลิสต์ว่างให้คืน {})"""
    # TODO
    raise NotImplementedError


def my_calibration_error(reclist, user_profile, genre_of):
    # TODO
    raise NotImplementedError


def my_parity_gap(scores_by_group):
    """scores_by_group : {ชื่อกลุ่ม: [คะแนนรายผู้ใช้]}"""
    # TODO
    raise NotImplementedError


def precision5(model_dict, user):
    """Precision@5 ของผู้ใช้หนึ่งคน เทียบกับ RELEVANT8"""
    # TODO
    raise NotImplementedError


# --------------------------- ตรวจคำตอบ ---------------------------
print("โปรไฟล์แนวของ Sophia :", {k: round(v, 4) for k, v in
                                 genre_distribution(PROFILE8["Sophia"], GENRE).items()})
print("แนวที่ Model X แนะนำ  :", {k: round(v, 4) for k, v in
                                 genre_distribution(MODEL_X["Sophia"], GENRE).items()})
print("แนวที่ Model Y แนะนำ  :", {k: round(v, 4) for k, v in
                                 genre_distribution(MODEL_Y["Sophia"], GENRE).items()})
print()

check("10.1 สัดส่วนรวมเป็น 1.0",
      sum(genre_distribution(MODEL_X["Emma"], GENRE).values()), 1.0000)
check("10.2 CalibErr ของ Sophia กับ Model X",
      my_calibration_error(MODEL_X["Sophia"], PROFILE8["Sophia"], GENRE), 0.6000)
check("10.3 CalibErr ของ Sophia กับ Model Y",
      my_calibration_error(MODEL_Y["Sophia"], PROFILE8["Sophia"], GENRE), 0.0000,
      hint="Model Y แนะนำสัดส่วนแนวตรงกับประวัติของ Sophia พอดี")

pk_x = {u: precision5(MODEL_X, u) for u in USERS8}
pk_y = {u: precision5(MODEL_Y, u) for u in USERS8}
grp_x = {g: [pk_x[u] for u in us] for g, us in USER_GROUP.items()}
grp_y = {g: [pk_y[u] for u in us] for g, us in USER_GROUP.items()}

print(f"{'กลุ่ม':<14}{'P@5 ของ Model X':>20}{'P@5 ของ Model Y':>20}")
print("-" * 54)
for g in USER_GROUP:
    print(f"{g:<14}{np.mean(grp_x[g]):>20.4f}{np.mean(grp_y[g]):>20.4f}")
print()

check("10.4 P@5 ของ Emma กับ Model Y", pk_y["Emma"], 0.8000)
check("10.5 ParityGap ของ Model X", my_parity_gap(grp_x), 0.0000)
check("10.6 ParityGap ของ Model Y", my_parity_gap(grp_y), 0.2000)
check("10.7 P@5 เฉลี่ยของ Y สูงกว่า X ในทุกกลุ่ม",
      all(np.mean(grp_y[g]) > np.mean(grp_x[g]) for g in USER_GROUP), True)

print("บทเรียน Leveling Down")
print(f"  Model X ได้ ParityGap = {my_parity_gap(grp_x):.4f} ซึ่งดู 'เป็นธรรมสมบูรณ์'")
print(f"  Model Y ได้ {my_parity_gap(grp_y):.4f} ทั้งที่ดีกว่าใน *ทุกกลุ่ม*")
print("  เกณฑ์ตายตัว 'Gap ≤ 0.05' จะปฏิเสธ Model Y ทันที -> ต้องรายงานค่าเฉลี่ยรายกลุ่มด้วยเสมอ")

---
## ข้อ 11 — Temporal Split และการตรวจการรั่วไหล ⭐⭐⭐
**หัวข้อ 2.8**

ใช้ชุดข้อมูล `INTERACTIONS` (มีคอลัมน์ `user_id`, `item_id`, `rating`, `timestamp`, `week`)

**สิ่งที่ต้องเขียน**

1. `my_global_temporal_split(df, train_frac)` — เรียงตามเวลา แล้วตัดที่จุดเดียวทั้งระบบ
2. `my_random_split(df, test_frac, seed)` — สุ่มโดยไม่สนใจเวลา (**ตัวอย่างของสิ่งที่ไม่ควรทำ**)
3. `my_leakage_report(train, test)` — นับเหตุการณ์ในชุดฝึกที่เกิด **หลัง** เหตุการณ์ที่เก่าที่สุดในชุดทดสอบ

**สิ่งที่จะได้เห็น** การแบ่งตามเวลาต้องได้ `n_future_in_train = 0` พอดี
ส่วนการแบ่งแบบสุ่มจะได้ค่าที่เกือบเท่ากับขนาดชุดฝึกทั้งหมด — ซึ่งเป็นไปไม่ได้ในระบบจริง

> **วินัยที่ป้องกันได้ข้อเดียว** การแบ่งข้อมูลต้องเป็นคำสั่งแรกเสมอ
> ก่อนคำนวณสถิติใด ๆ แม้แต่ค่าเฉลี่ยหรือพจนานุกรมรายการ

In [ ]:
# ============================== ข้อ 11 ==============================
def my_global_temporal_split(df, train_frac=0.8):
    """เรียงตาม timestamp แล้วตัดที่จุดเดียว คืน (train, test)"""
    # TODO
    raise NotImplementedError


def my_random_split(df, test_frac=0.2, seed=42):
    """สุ่มโดยไม่สนใจเวลา — ตัวอย่างของสิ่งที่ไม่ควรทำ  คืน (train, test)"""
    # TODO: ใช้ np.random.default_rng(seed).permutation(len(df))
    raise NotImplementedError


def my_leakage_report(train, test):
    """คืน dict อย่างน้อยสองคีย์

    n_future_in_train : จำนวนแถวในชุดฝึกที่ timestamp > timestamp ต่ำสุดของชุดทดสอบ
    verdict           : "ปลอดภัย" ถ้าเท่ากับ 0 มิฉะนั้น "พบการรั่วไหล"
    """
    # TODO
    raise NotImplementedError


# --------------------------- ตรวจคำตอบ ---------------------------
tr_t, te_t = my_global_temporal_split(INTERACTIONS, 0.8)
tr_r, te_r = my_random_split(INTERACTIONS, 0.2, seed=42)
rep_t = my_leakage_report(tr_t, te_t)
rep_r = my_leakage_report(tr_r, te_r)

print(f"{'วิธีแบ่ง':<26}{'ฝึก':>9}{'ทดสอบ':>9}{'อนาคตในชุดฝึก':>20}{'ผลตรวจ':>18}")
print("-" * 84)
for nm, tr, te, rp in [("ตามเวลาแบบรวม", tr_t, te_t, rep_t), ("สุ่ม (Random)", tr_r, te_r, rep_r)]:
    print(f"{nm:<26}{len(tr):>9,}{len(te):>9,}"
          f"{rp['n_future_in_train']:>13,} ({rp['n_future_in_train']/len(tr)*100:>4.1f}%)"
          f"{rp['verdict']:>18}")
print()

n_total = len(INTERACTIONS)
check("11.1 ขนาดชุดฝึกของ temporal split", len(tr_t), int(n_total * 0.8))
check("11.2 ชุดฝึก+ชุดทดสอบ = ข้อมูลทั้งหมด", len(tr_t) + len(te_t), n_total)
check("11.3 temporal split ไม่มีอนาคตในชุดฝึก", rep_t["n_future_in_train"], 0)
check("11.4 temporal split ได้ผลตรวจ 'ปลอดภัย'", rep_t["verdict"], "ปลอดภัย")
check("11.5 เวลาสูงสุดของชุดฝึก ≤ เวลาต่ำสุดของชุดทดสอบ",
      tr_t.timestamp.max() <= te_t.timestamp.min(), True)
check("11.6 random split พบการรั่วไหล", rep_r["verdict"], "พบการรั่วไหล")
check("11.7 random split รั่วเกิน 90% ของชุดฝึก",
      rep_r["n_future_in_train"] / len(tr_r) > 0.90, True,
      hint="เพราะเหตุการณ์ที่เก่าที่สุดของชุดทดสอบอยู่ใกล้จุดเริ่มต้นของข้อมูล")
check("11.8 random split ใช้ seed เดิมแล้วได้ผลเดิม",
      len(my_random_split(INTERACTIONS, 0.2, seed=42)[1]), len(te_r))

print("ข้อสรุป: เปลี่ยนเพียงวิธีแบ่งข้อมูล โดยไม่แตะแบบจำลองเลย")
print("        ก็เพียงพอที่จะทำให้ตัวเลขที่รายงานสูงเกินจริงอย่างเป็นระบบ")

---
## ข้อ 12 — Sampled Negatives และ Winner Flip ⭐⭐⭐
**หัวข้อ 2.9 — หัวข้อที่สำคัญที่สุดของทั้งบท**

เมื่อสุ่มรายการเชิงลบ $m$ ชิ้นจากคลังขนาด $|\mathcal{I}|$ อันดับที่สังเกตได้มีค่าคาดหวัง

$$\mathbb{E}[\tilde{\rho}]\approx 1+(\rho-1)\cdot\frac{m}{|\mathcal{I}|-1}$$

**สิ่งที่ต้องเขียน**

1. `my_ndcg_from_rank(rank, K)` — nDCG@K เมื่อมีของถูกเพียงหนึ่งชิ้นที่อันดับ `rank`
   (IDCG = 1 ดังนั้น nDCG = DCG และถ้า `rank > K` ต้องได้ 0 ทันที — **จุดตัดแข็ง**)
2. `expected_sampled_rank(true_rank, n_items, m)` — สูตรข้างบน
3. `my_sampled_rank(true_rank, n_items, m, rng)` — จำลองด้วยการแจกแจงไฮเพอร์จีออเมตริก
   (`rng.hypergeometric(ngood, nbad, nsample)`)

**สุดท้าย** หาค่า $m$ ที่ทำให้เกิด **Winner Flip** ระหว่างสองแบบจำลอง

* Model P — อันดับจริง = 12 สำหรับผู้ใช้ทุกคน (สม่ำเสมอ แต่หลุด Top-10 เสมอ)
* Model Q — อันดับ 3 สำหรับผู้ใช้ 30% และอันดับ 60 สำหรับ 70%

คลัง 500 รายการ ผู้ใช้ 800 คน K = 10 — **แบบจำลองใดดีกว่าสำหรับระบบที่แสดงสิบอันดับแรก**

In [ ]:
# ============================== ข้อ 12 ==============================
def my_ndcg_from_rank(rank, K=10):
    """nDCG@K เมื่อมีรายการเกี่ยวข้องเพียงหนึ่งชิ้น ณ อันดับ rank"""
    # TODO
    raise NotImplementedError


def expected_sampled_rank(true_rank, n_items, m):
    """อันดับคาดหวังตามสูตร  E[ρ~] = 1 + (ρ − 1)·m/(|I| − 1)"""
    # TODO
    raise NotImplementedError


def my_sampled_rank(true_rank, n_items, m, rng):
    """อันดับที่สังเกตได้เมื่อสุ่มรายการเชิงลบ m ชิ้นแบบสม่ำเสมอ

    ของที่ 'เหนือกว่า' เป้าหมายมี (true_rank − 1) ชิ้น จากรายการเชิงลบทั้งหมด (n_items − 1)
    ใช้ rng.hypergeometric(ngood=..., nbad=..., nsample=...)
    อันดับใหม่ = 1 + จำนวนที่เหนือกว่าและถูกสุ่มติดมา
    """
    # TODO
    raise NotImplementedError


# TODO: หาค่า m ที่ทำให้ Model P ชนะ Model Q (ตอบเป็นจำนวนเต็ม)
m_where_P_wins = None

# --------------------------- ตรวจคำตอบ ---------------------------
check("12.1 อันดับ 1 -> nDCG@10 = 1.0", my_ndcg_from_rank(1, 10), 1.0000)
check("12.2 อันดับ 10 -> nDCG@10", my_ndcg_from_rank(10, 10), 0.2891)
check("12.3 อันดับ 11 -> nDCG@10 = 0 (จุดตัดแข็ง)", my_ndcg_from_rank(11, 10), 0.0000)
check("12.4 สูตรอันดับคาดหวัง (ρ=500, |I|=5000, m=99)",
      expected_sampled_rank(500, 5000, 99), 10.8822)

rng = np.random.default_rng(2569)
sim = np.array([my_sampled_rank(500, 5000, 99, rng) for _ in range(20000)])
check("12.5 การจำลองตรงกับทฤษฎี (คลาดเคลื่อน < 0.3)",
      abs(sim.mean() - expected_sampled_rank(500, 5000, 99)) < 0.3, True)
check("12.6 sampled nDCG@10 ของอันดับจริง 500 อยู่ราว 0.15",
      abs(np.mean([my_ndcg_from_rank(int(r), 10) for r in sim]) - 0.1482) < 0.02, True)

# ---- Winner Flip
N_ITEMS, N_USERS, K = 500, 800, 10
ranks_P = np.full(N_USERS, 12)
ranks_Q = np.concatenate([np.full(int(0.30 * N_USERS), 3), np.full(N_USERS - int(0.30 * N_USERS), 60)])


def mean_sampled_ndcg(ranks, m, rng, n_items=N_ITEMS, K=K):
    if m is None:
        return float(np.mean([my_ndcg_from_rank(int(r), K) for r in ranks]))
    return float(np.mean([my_ndcg_from_rank(my_sampled_rank(int(r), n_items, m, rng), K)
                          for r in ranks]))


rng = np.random.default_rng(7)
print(f"\n{'m':>16}{'nDCG@10 ของ P':>18}{'nDCG@10 ของ Q':>18}{'ผู้ชนะ':>10}")
print("-" * 64)
for m in [20, 50, 100, 200, None]:
    p = mean_sampled_ndcg(ranks_P, m, rng)
    q = mean_sampled_ndcg(ranks_Q, m, rng)
    tag = "เต็มคลัง" if m is None else str(m)
    print(f"{tag:>16}{p:>18.4f}{q:>18.4f}{('P' if p > q else 'Q'):>10}")

p_full = mean_sampled_ndcg(ranks_P, None, rng)
q_full = mean_sampled_ndcg(ranks_Q, None, rng)
check("12.7 เต็มคลัง: P ได้ศูนย์สนิท", p_full, 0.0000)
check("12.8 เต็มคลัง: Q ชนะ P", q_full > p_full, True)
if m_where_P_wins is not None:
    rng2 = np.random.default_rng(7)
    p_m = mean_sampled_ndcg(ranks_P, m_where_P_wins, rng2)
    q_m = mean_sampled_ndcg(ranks_Q, m_where_P_wins, rng2)
    check(f"12.9 ที่ m = {m_where_P_wins} แล้ว P ชนะ Q", p_m > q_m, True,
          hint="ลองค่า m ที่ใช้แพร่หลายที่สุดในวรรณกรรม")
else:
    check("12.9 ระบุค่า m ที่ทำให้ P ชนะ", False, True, hint="เติมตัวแปร m_where_P_wins")

print("\nผู้ชนะสลับกันได้โดยไม่เปลี่ยนพารามิเตอร์ของแบบจำลองแม้แต่ตัวเดียว")
print("เพราะ nDCG@K มีจุดตัดแข็งที่ K: อันดับ 12 ของ P ถูกการสุ่ม 'อุ้ม' เข้า Top-10 อย่างสม่ำเสมอ")
print("ขณะที่อันดับ 60 ของผู้ใช้ 70% ของ Q ไกลเกินกว่าจะถูกอุ้ม (Krichene & Rendle, KDD 2020)")

---
## ★ โจทย์ท้าทาย — สร้างแบบจำลองของตัวเอง ⭐⭐⭐⭐
**หัวข้อ 2.6 + 2.10 · ไม่มีเฉลยตายตัว ตรวจด้วยเกณฑ์ ไม่ใช่ด้วยค่าที่ถูกต้องเพียงค่าเดียว**

### โจทย์

ออกแบบฟังก์ชัน `my_ranker(user_id, candidates)` ที่คืนลิสต์รายการเรียงอันดับ
โดยต้องทำให้ผ่าน **ทั้งสามเงื่อนไข** พร้อมกัน เทียบกับเส้นฐาน `TopPopular`

| เงื่อนไข | เกณฑ์ |
|---|---|
| Primary Metric | `nDCG@10` **สูงกว่า** TopPopular |
| Guardrail 1 | `Coverage@10` **สูงกว่า** TopPopular |
| Guardrail 2 | `Novelty` **ไม่ต่ำกว่า** TopPopular |

### ข้อบังคับเชิงระเบียบวิธี

* ห้ามใช้ข้อมูลจาก `test` ในการสร้างแบบจำลองไม่ว่ากรณีใด (ทั้งความนิยม ค่าเฉลี่ย หรือรายชื่อรายการ)
* สถิติทุกตัวต้องคำนวณจาก `train` เท่านั้น
* ต้องรายงานผลพร้อม paired bootstrap ระดับผู้ใช้

### ใบ้

* เส้นฐานความนิยมล้วนจะกระจุกที่รายการเดิม ๆ — ลองผสมสัญญาณอื่นเข้าไป
* การ "ลงโทษ" รายการที่ถูกแนะนำไปแล้วหลายครั้ง ช่วยดัน Coverage ขึ้นได้
* ระวังไม่ให้ nDCG ตกจนแลกไม่คุ้ม — นี่คือการต่อรองบน Pareto Frontier ตัวจริง

In [ ]:
# ============================== โจทย์ท้าทาย ==============================
# สนามทดลอง: ใช้ INTERACTIONS แบ่งตามเวลา 80/20 (ใช้ฟังก์ชันที่เขียนไว้ในข้อ 11)
train_b, test_b = my_global_temporal_split(INTERACTIONS, 0.8)
all_items_b = sorted(INTERACTIONS.item_id.unique())
seen_b = train_b.groupby("user_id")["item_id"].apply(set).to_dict()
rel_b = test_b[test_b.rating >= 4].groupby("user_id")["item_id"].apply(set).to_dict()
pop_b = Counter(train_b.item_id)
pop_order_b = [it for it, _ in pop_b.most_common()]
n_users_b = train_b.user_id.nunique()


def baseline_toppopular(user_id, candidates):
    cs = set(candidates)
    head = [i for i in pop_order_b if i in cs]
    hs = set(head)
    return head + [i for i in candidates if i not in hs]


def my_ranker(user_id, candidates):
    """TODO: เขียนแบบจำลองของคุณที่นี่ (ห้ามแตะ test_b / rel_b)"""
    return baseline_toppopular(user_id, candidates)     # <-- แทนที่ด้วยของคุณ


# --------------------------- ประเมินและตรวจเกณฑ์ ---------------------------
def bench(rank_fn, K=10):
    nd, lists, nov = [], [], []
    scores = {}
    for u, items in rel_b.items():
        cands = [i for i in all_items_b if i not in seen_b.get(u, set())]
        if not items & set(cands):
            continue
        top = rank_fn(u, cands)[:K]
        s = my_ndcg_at_k(top, {i: 1.0 for i in items}, K)
        nd.append(s); scores[u] = s; lists.append(top)
        nov.append(my_novelty(top, dict(pop_b), n_users_b))
    shown = {x for l in lists for x in l}
    return {"nDCG@10": float(np.mean(nd)),
            "Coverage@10": len(shown) / len(all_items_b),
            "Novelty": float(np.mean(nov))}, scores


base_res, base_scores = bench(baseline_toppopular)
mine_res, mine_scores = bench(my_ranker)
display(pd.DataFrame({"TopPopular (เส้นฐาน)": base_res, "แบบจำลองของคุณ": mine_res}).T.round(4))

c1 = mine_res["nDCG@10"] > base_res["nDCG@10"]
c2 = mine_res["Coverage@10"] > base_res["Coverage@10"]
c3 = mine_res["Novelty"] >= base_res["Novelty"]
print(f"\nPrimary  nDCG@10 สูงกว่าเส้นฐาน      : {'✅' if c1 else '❌'}")
print(f"Guard 1  Coverage@10 สูงกว่าเส้นฐาน  : {'✅' if c2 else '❌'}")
print(f"Guard 2  Novelty ไม่ต่ำกว่าเส้นฐาน    : {'✅' if c3 else '❌'}")
check("★ ผ่านครบทั้งสามเงื่อนไข", c1 and c2 and c3, True,
      hint="ลองผสมความนิยมกับการลงโทษรายการที่ถูกแนะนำซ้ำ")

# ---- รายงานความไม่แน่นอนแบบย่อ (paired bootstrap ระดับผู้ใช้)
common = sorted(set(base_scores) & set(mine_scores))
d = np.array([mine_scores[u] - base_scores[u] for u in common])
rngb = np.random.default_rng(11)
boots = d[rngb.integers(0, len(d), size=(4000, len(d)))].mean(axis=1)
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"\nส่วนต่างเฉลี่ยรายผู้ใช้ = {d.mean():+.4f}   CI 95% = [{lo:+.4f}, {hi:+.4f}]")
print(f"ผู้ใช้ที่ได้ผลแย่ลง = {(d < 0).mean()*100:.1f}%   "
      f"ข้อสรุป: {'มีนัยสำคัญ' if (lo > 0 or hi < 0) else 'ไม่มีหลักฐานว่าต่างกัน'}")

---
## สรุปคะแนน

รันเซลล์ถัดไปเพื่อดูว่าผ่านกี่ข้อ และข้อใดยังไม่ผ่าน

In [ ]:
show_score()

---
## เมื่อทำครบแล้ว

### คำถามทบทวนที่ไม่มีเซลล์ตรวจ (ตอบด้วยการเขียน)

1. ทีม ก รายงาน `Precision@10` ทีม ข รายงาน `MRR` บนข้อมูลชุดเดียวกัน แล้วสรุปคนละอย่าง
   ทั้งสองทีมคำนวณถูกทุกหลัก — **จะป้องกันสถานการณ์นี้อย่างไร**
2. ระบบใหม่ได้ `nDCG@10` สูงขึ้น 13.9% แต่เวลาใช้งานจริงลดลง 2.1%
   **ตัวชี้วัดกำกับตัวใดที่ถ้ากำหนดไว้ล่วงหน้า จะปฏิเสธระบบนี้ได้ตั้งแต่ขั้นออฟไลน์**
3. เพื่อนร่วมทีมเสนอว่า "ใช้ m = 100 เหมือนงานตีพิมพ์อื่นก็ได้ จะได้เทียบกันได้"
   **ข้อเสนอนี้ผิดตรงไหน**
4. ผลบอกว่า A ดีกว่า B โดยเฉลี่ย แต่ผู้ใช้ 40% ได้ผลแย่ลง **ควรเปลี่ยนระบบหรือไม่ เพราะอะไร**
5. `Coverage@10` ของระบบเราชนเพดานที่ 1.0 แล้ว **ตัวชี้วัดนี้ยังบอกอะไรเราได้อีกหรือไม่**

### ไฟล์ประกอบ

* `Lab2_All_in_One.ipynb` — เนื้อหาและการทดลองครบทั้งบท รวมส่วนเปรียบเทียบแบบจำลอง 5 ตัว
* `Lab2_Exercises_Solutions.ipynb` — เฉลยพร้อมคำอธิบายทีละข้อ